# Forward curves (rates and commodity-style projections)

Deep-dive reference: **projection-factor curves** for a floating-rate index with a fixed **index tenor** (e.g. 3M SOFR), contrasted with commodity forward **price** curves.

## Concept

A **forward curve** stores **projection factors** \(P_x(t)\) for one floating-rate index, with \(P_x(0) = 1\). The growth \(P_x(t_1)/P_x(t_2)\) over an index period \([t_1, t_2]\) sets the projected fixing \(L = (P_x(t_1)/P_x(t_2) - 1)/\alpha\), where \(\alpha\) is the period's accrual fraction under the index's contractual day count. The curve's `index_tenor` (e.g. `3M`) records the index period length. In rates, this supports floating coupons and curve-consistent projections. Commodity forward **prices** are levels rather than growth factors, so they live on `PriceCurve`.

## API walkthrough

`ForwardCurve(options)` takes a dictionary with curve metadata and `(time_years, projection_factor)` knots; `(0.0, 1.0)` is prepended when absent. `rate_between(t1, t2, accrual_year_fraction)` returns the simple fixing over `[t1, t2]`, `rate_between_dates(start, end, accrual_year_fraction)` does the same from dates, `instantaneous_forward(t)` returns the continuously compounded forward, and `df(t)` returns the factor itself.

The knots below encode constant continuous forwards of 4.25% over the first year and 4.75% from 1Y to 5Y; `log_linear` interpolation keeps the forward constant between knots, and `extend` carries the last segment's forward past 5Y. On an `act_360` curve a 0.25-year step is a 90-day index period on the same basis, so `0.25` is also its accrual fraction. A constant continuous forward `r` projects the simple fixing `expm1(r * a) / a` over accrual `a`, slightly above `r`.

In [ ]:
import math
from datetime import date

from finstack_quant.core.dates import DayCount
from finstack_quant.core.market_data import ForwardCurve

fwd = ForwardCurve(
    {
        "id": "USD-SOFR-3M",
        "base_date": date(2024, 1, 1),
        "knots": [(1.0, math.exp(-0.0425 * 1.0)), (5.0, math.exp(-0.0425 * 1.0 - 0.0475 * 4.0))],
        "interp": "log_linear",
        "extrapolation": "extend",
        "day_count": "act_360",
        "index_tenor": "3M",
    }
)
print("curve:", fwd)
print("index tenor:", fwd.index_tenor)
print("knots -> projection factors:", dict(zip(fwd.knots, fwd.factors)))
print("projected 3M fixing starting at 1Y: rate_between(1.0, 1.25, 0.25) =", fwd.rate_between(1.0, 1.25, 0.25))
for t in (0.0, 0.5, 1.0, 3.0, 5.0):
    print(f"  t={t}y -> 3M fixing {fwd.rate_between(t, t + 0.25, 0.25):.6f}  mid-period continuous forward {fwd.instantaneous_forward(t + 0.125):.6f}")
assert abs(fwd.rate_between(0.5, 0.75, 0.25) - math.expm1(0.0425 * 0.25) / 0.25) < 1e-12

# Dated fixing: the index tenor sets the period end, the index day count sets the accrual.
start = date(2025, 4, 2)
end = fwd.index_tenor.add_to_date(start, business_day_convention="unadjusted")
accrual = DayCount.ACT_360.year_fraction(start, end)
print(f"3M fixing {start} -> {end} (accrual {accrual:.6f}): {fwd.rate_between_dates(start, end, accrual):.6f}")

Link to a **discount curve**: over an interval \([t_1, t_2]\), the continuously compounded forward implied by discount factors is \(\ln(DF(t_1)/DF(t_2))/(t_2-t_1)\). A projection curve applies the same algebra to its own factors, and in multi-curve setups the **index projection curve** is calibrated separately from the OIS discount curve. `ForwardCurve.from_discount_curve` promotes a discount curve into a projection curve for single-curve setups, so its fixings reproduce the discount-implied forwards.

In [ ]:
import math
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, ForwardCurve

disc = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (1.0, 0.96), (2.0, 0.92)],
    interp="monotone_convex",
    extrapolation="extend",
    day_count="act_365f",
)
t1, t2 = 1.0, 2.0
fwd_from_df = disc.forward(t1, t2)
print(f"Discount-implied continuous forward {t1}y -> {t2}y: {fwd_from_df:.6f}")

single_curve = ForwardCurve.from_discount_curve("USD-OIS-1Y", disc, "1Y")
simple_fixing = single_curve.rate_between(t1, t2, t2 - t1)
assert abs(math.log1p(simple_fixing * (t2 - t1)) / (t2 - t1) - fwd_from_df) < 1e-12
print(f"Single-curve 1Y fixing {t1}y -> {t2}y: {simple_fixing:.6f} (simple, same growth as the DF ratio)")

sofr_3m = ForwardCurve(
    {
        "id": "USD-SOFR-3M",
        "base_date": date(2024, 1, 1),
        "knots": [(1.0, math.exp(-0.043 * 1.0)), (2.0, math.exp(-0.043 * 1.0 - 0.045 * 1.0))],
        "interp": "log_linear",
        "extrapolation": "extend",
        "day_count": "act_360",
        "index_tenor": "3M",
    }
)
print(f"SOFR-3M projected fixing at t={t1}y: {sofr_3m.rate_between(t1, t1 + 0.25, 0.25):.6f} (index projection, not DF parity check)")

## Practical example

**Commodity use case:** forward **prices** (USD/bbl) at future times are price levels, not projection factors, so they live on `PriceCurve`; `price(t)` interpolates the level at horizon `t`.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import PriceCurve

# Illustrative forward prices along time; price(t) is the curve level at t.
com_fwd = PriceCurve({
    "id": "WTI-PROJ",
    "base_date": date(2024, 1, 1),
    "knots": [(0.0, 74.0), (0.5, 73.0), (1.0, 72.0), (2.0, 71.0)],
    "interp": "linear",
    "extrapolation": "flat",
    "day_count": "act_365f",
    "kind": "price",
})
print("Commodity forward prices ($/bbl):")
for t in (0.0, 0.25, 1.0):
    print(f"  at {t}y: {com_fwd.price(t):.2f}")

## Takeaways

- **Projection factors** \(P_x(t)\) with \(P_x(0) = 1\) are the stored state; **`rate_between(t1, t2, accrual)`** turns factor growth into the simple fixing on the index's contractual accrual fraction, **`rate_between_dates`** does the same from dates, and **`instantaneous_forward(t)`** gives the continuous forward.
- **`index_tenor`** records the index period (e.g. `3M`); add it to a start date to get the fixing period end. A constant continuous forward `r` projects `expm1(r * a) / a` over accrual `a`, not `r`.
- **Discount curves** give **implied forwards** via `forward(t1, t2)`; `ForwardCurve.from_discount_curve` covers single-curve setups, while production desks calibrate separate **index projection** curves jointly with OIS discounting.
- **Commodities:** forward **price** term structures belong on **`PriceCurve`**, not on a projection-factor curve.

## Multi-curve calibration (OIS discount + index forward)

Production rates desks maintain separate curves for discounting (OIS) and index projection (e.g. 3M SOFR). A **multi-step calibration plan** bootstraps the discount curve first, then uses it as the discounting backbone for the forward curve step.

The forward step references the discount curve via `"discount_curve_id"` — the engine automatically feeds the discount curve produced in step 1 into step 2.

The quote strip below is intentionally **internally consistent** and includes a **3Y forward-swap anchor** so the long end remains well behaved. The example also **fails closed**: if calibration does not succeed, it raises immediately instead of printing misleading forward levels.

In [ ]:
import sys
sys.path.insert(0, "../..")

from _shared import REPOSITORY_ROOT
from finstack_quant.calibration import calibrate

envelope_json = (
    REPOSITORY_ROOT
    / "finstack-quant/calibration/examples/market_bootstrap/02_usd_3m_forward_curve.json"
).read_text()

result = calibrate(envelope_json)
if not result.success:
    raise RuntimeError(result.to_dataframe().to_string(index=False))
print("Success:", result.success)
print()

disc = result.market.get_discount("USD-OIS")
fwd = result.market.get_forward("USD-SOFR-3M")

# The calibrated SOFR curve is act_360, so a 0.25-year step is a 90-day 3M index period.
print("Index tenor:", fwd.index_tenor, "| day count:", fwd.day_count)
print("Tenor (y)   OIS DF       OIS zero     3M fwd rate")
for t in [0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    print(f"  {t:5.2f}     {disc.df(t):.8f}  {disc.zero_rate(t, 'continuous'):.6f}   {fwd.rate_between(t, t + 0.25, 0.25):.6f}")


In [ ]:
# Per-step calibration quality
print(result.to_dataframe().to_string(index=False))